# HumanInTheLoopMiddleware 中间件

`HumanInTheLoopMiddleware`（人在环中间件，HITL）用来为 Agent 的工具调用添加人工审核。更准确地说，它是一个 `after_model` 钩子：

1. 模型先生成 tool calls。

2. 中间件检查每个调用是否命中 `interrupt_on` 策略。

3. 命中时，在**工具真正执行前**发出 interrupt，保存状态并暂停。

4. 人工给出决定后，图从暂停点继续。

这类中断适合发送邮件、执行 SQL 写入、删除数据、资金操作等带有副作用或高风险的行为。

## 参数说明

### `interrupt_on`：工具名与审批策略的映射

策略值可以是 `True`、`False` 或 `InterruptOnConfig`字典。

```python
interrupt_on={
    "get_weather": True,
    "read_email_tool": False,
    "send_email_tool": {
        "allowed_decisions": ["approve", "reject"],
    },
}
```

1. `True`：该工具需要审核，默认允许 `approve`、`edit`、`reject`、`respond` 四种决定。

2. `False`：不中断，工具自动执行。

3. `InterruptOnConfig`：精细控制某个工具。其中 `allowed_decisions` 控制允许的决策；`description` 可以是字符串或根据本次参数生成描述的函数；`when` 则可以根据工具参数条件性地决定是否中断。

### `description_prefix`：默认审批说明

`description_prefix` 用来修改所有工具审批请求的通用描述。如果单个工具配置了 `description`，它优先于 `description_prefix`。


## 制造中断（兼容模式示例）

HITL 依赖 LangGraph 的 checkpoint 机制保存暂停位置和当前状态。因此：

- Agent 必须配置 `checkpointer`。

- 第一次调用和恢复调用必须使用同一个 `thread_id`。

- `InMemorySaver` 适合学习和单进程测试；生产中应使用持久化 checkpointer。


下面的主示例有意保留默认兼容格式：中断从 `response["__interrupt__"]` 读取。后面会给出推荐的 `version="v2"` 写法。


In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint

@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res

@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"

@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID: {email_id}\n是空的"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是: {subject}, 内容: {body}"

agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断啦" # 这个消息是给用户看的
                },
            },
            description_prefix="中断啦"
        ),
    ],
)

config = {"configurable": {"thread_id": "1"}}

# 下面故意不传 version="v2"，用于对比默认兼容格式。
# 第一次调用：遇到需审批的工具调用时会暂停
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天北京的天气"
                              "查询今日新闻"
                              "查看ID为 'sk2131421' 的邮件内容，"
                              "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                              "同时做这四件事")
        ]
    },
    config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)


# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("===== interrupts =====")
# print(interrupts)

# ===== 逐个打印 interrupt 请求 =====
# 模型如果没有调用命中策略的工具，interrupts 会为空。
action_requests = []
if interrupts:
    action_requests = interrupts[0].value["action_requests"]
    for action_request in action_requests:
        rprint(action_request)
else:
    print("本次未触发中断：模型可能没有调用需审批的工具。")

==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='440fc231-cc53-4198-9203-8c71ea1b9e1b'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 100,
                    'prompt_tokens': 199,
                    'total_tokens': 299,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00059925,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00059925,
                        'upstream_inference_prompt_cost': 0.00014925,
                        'upstream_inference_completions_cost': 0.00045
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784629246-5fJoHsO03iBMesVGvWJp',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f8431-0bb8-7170-9a01-7c3a86836e21-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': True},
                    'id': 'call_MRTmLfQ4i8lIEWVr0brw0pdI',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_99pkvYtxXwpKLyPA7AlKp9SI', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_dwtYMN8FNpx7T15H21CP8roh',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_XWOhhLSCgzVZEZJPVmD5UUgu',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 199,
                'output_tokens': 100,
                'total_tokens': 299,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': True},
                        'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': True}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断啦'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'al

===== interrupts =====


{
    'name': 'get_weather',
    'args': {'city': '北京', 'is_forcast': True},
    'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': True}"
}

{'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
    'description': '发送邮件中断啦'
}

## 推荐写法：`version="v2"`

默认兼容模式中，`invoke()` 返回普通字典，中断位于 `result["__interrupt__"]`。

`version="v2"` 中，返回值是 `GraphOutput`：

- `result.value`：Agent 状态（例如 `messages`）。

- `result.interrupts`：需要人工决定的 `Interrupt` 元组。


对新代码，建议显式传入 `version="v2"`，不再依赖 `__interrupt__` 这个兼容字段。


In [ ]:
# 使用上个单元格创建的 agent，但使用推荐的 v2 返回格式。
v2_config = {"configurable": {"thread_id": "hitl-v2-demo"}}

first_result = agent.invoke(
    {"messages": [HumanMessage(content="请查询北京天气")]},
    config=v2_config,
    version="v2",
)

# v2 下直接读属性，不再读 result["__interrupt__"]。
if first_result.interrupts:
    interrupt_value = first_result.interrupts[0].value
    rprint(interrupt_value["action_requests"])

    final_result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=v2_config,  # 必须与暂停时使用同一个 thread_id
        version="v2",
    )
    final_result.value["messages"][-1].pretty_print()
else:
    print("本次没有触发中断。")


## `description`：把审核提示写成人能看懂的话

`description` 可以是静态字符串，也可以是接收 `ToolCallRequest` 的函数。后者可以根据本次的收件人、SQL或文件路径生成审批斎要。单个工具的 `description` 会覆盖全局 `description_prefix`。


In [ ]:
from langchain.agents.middleware import ToolCallRequest


def email_review_description(request: ToolCallRequest) -> str:
    args = request.tool_call["args"]
    return (
        "请审核即将发送的邮件："
        f"收件人={args.get('recipient')}，主题={args.get('subject')}"
    )


description_demo_middleware = HumanInTheLoopMiddleware(
interrupt_on={
    "send_email_tool": {
        "allowed_decisions": ["approve", "edit", "reject"],
        # 可以写固定字符串，也可以像这样依据参数生成。
        "description": email_review_description,
    }
}
)


## `when`：只审批真正有风险的参数

`when` 是一个判断函数：返回 `True` 则中断审批，返回 `False` 则自动执行。例如，给公司内部邮箱的邮件可自动执行，对外发送才需要人工审核。`when` 需要 `langchain >= 1.3.3`。


In [ ]:
def is_external_email(request: ToolCallRequest) -> bool:
    recipient = request.tool_call["args"].get("recipient", "")
    return not recipient.endswith("@mycompany.com")


conditional_hitl_middleware = HumanInTheLoopMiddleware(
interrupt_on={
    "send_email_tool": {
        "allowed_decisions": ["approve", "edit", "reject"],
        "when": is_external_email,
    }
}
)

# recipient="alice@mycompany.com" -> when 返回 False，不中断。
# recipient="customer@example.com" -> when 返回 True，中断等待审批。


## `respond`：人是这个工具的“返回值提供者”

`respond` 不执行原工具，而是把人工的 `message` 直接作为一条成功的 `ToolMessage` 返回给 Agent。它适合 `ask_user`、索取密码、请用户选择方案等工具。

不要用 `respond` 拒绝发邮件、删除数据等工具：它会让模型认为该工具“已成功执行”。需要否决时应使用 `reject`。


In [ ]:
@tool
def ask_user(question: str) -> str:
    """向用户提出一个必须由人回答的问题。"""
    # 当人工选择 respond 时，这个函数不会被调用。
    raise RuntimeError("ask_user 应由 HumanInTheLoopMiddleware 的 respond 决定提供结果")


ask_user_agent = create_agent(
model=model,
tools=[ask_user],
middleware=[
    HumanInTheLoopMiddleware(
        interrupt_on={
            "ask_user": {"allowed_decisions": ["respond"]},
        }
    )
],
checkpointer=InMemorySaver(),
system_prompt="当需要用户选择时，必须调用 ask_user 工具。",
)

respond_config = {"configurable": {"thread_id": "hitl-respond-demo"}}
pending = ask_user_agent.invoke(
{"messages": [HumanMessage(content="请让用户在蓝色和红色之间选择一种主题色")]},
config=respond_config,
version="v2",
)

if pending.interrupts:
    completed = ask_user_agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "respond", "message": "蓝色"},
                ]
            }
        ),
        config=respond_config,
        version="v2",
    )
    completed.value["messages"][-1].pretty_print()


## 本例中 `Command(resume=...)` 的作用

本笔记中，`Command` 只用于一件事：**恢复被 HITL 暂停的这一次 Agent 执行**。

```python
Command(resume={"decisions": [...]})
```

它不是新的用户消息，也不会自己执行工具。LangGraph 先依据同一个 `thread_id` 找到 checkpoint 中的暂停点，再把 `resume` 载荷交给 `HumanInTheLoopMiddleware`。中间件根据决策列表依次执行：

- `approve`：以原参数执行工具。

- `edit`：以编辑后的参数执行工具。

- `reject`：不执行工具，向模型返回拒绝反馈。

- `respond`：不执行工具，把人工的消息作为工具结果返回。


`decisions` 的顺序必须与 interrupt 中 `action_requests` 的顺序一致。


In [2]:
# 如果有中断，说明进入人在环了
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "中国北京市通州区", "is_forcast": True}
    }
}

news_decision = {
    "type": "approve",
}

send_email_decision = {
    "type": "approve"
}

# 字典
decisions = {
    "decisions": [] # 字典列表
}

# 决策的顺序必须和返回的中断请求顺序一致
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

resumed_response = None

if interrupts:
    # 审批通过
    resumed_response = agent.invoke(
        Command(resume=decisions),  #Command 这里需要展开一些
        config=config,  # 必须是同一个 thread_id
    )

if resumed_response is not None:
    print("==== 审批后继续执行 ====")
    for msg in resumed_response["messages"]:
        msg.pretty_print()

    print("==== 审批后的原始响应 ====")
    rprint(resumed_response)
else:
    print("本次没有中断可恢复。")

>>> 真的执行发送邮件工具了
==== 审批后继续执行 ====
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_MRTmLfQ4i8lIEWVr0brw0pdI)
 Call ID: call_MRTmLfQ4i8lIEWVr0brw0pdI
  Args:
    city: 中国北京市通州区
    is_forcast: True
  get_news (call_99pkvYtxXwpKLyPA7AlKp9SI)
 Call ID: call_99pkvYtxXwpKLyPA7AlKp9SI
  Args:
  read_email_tool (call_dwtYMN8FNpx7T15H21CP8roh)
 Call ID: call_dwtYMN8FNpx7T15H21CP8roh
  Args:
    email_id: sk2131421
  send_email_tool (call_XWOhhLSCgzVZEZJPVmD5UUgu)
 Call ID: call_XWOhhLSCgzVZEZJPVmD5UUgu
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

中国北京市通州区今天天气不错
明天下雨
================================= Tool Message ======================

{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='440fc231-cc53-4198-9203-8c71ea1b9e1b'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 100,
                    'prompt_tokens': 199,
                    'total_tokens': 299,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00059925,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00059925,
                        'upstream_inference_prompt_cost': 0.00014925,
                        'upstream_inference_completions_cost': 0.00045
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784629246-5fJoHsO03iBMesVGvWJp',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f8431-0bb8-7170-9a01-7c3a86836e21-0',
            tool_calls=[
                {
                    'type': 'tool_call',
                    'name': 'get_weather',
                    'args': {'city': '中国北京市通州区', 'is_forcast': True},
                    'id': 'call_MRTmLfQ4i8lIEWVr0brw0pdI'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_99pkvYtxXwpKLyPA7AlKp9SI', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_dwtYMN8FNpx7T15H21CP8roh',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_XWOhhLSCgzVZEZJPVmD5UUgu',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 199,
                'output_tokens': 100,
                'total_tokens': 299,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='中国北京市通州区今天天气不错\n明天下雨',
            name='get_weather',
            id='9913c883-efdc-43fc-88d3-7a396b181e1d',
            tool_call_id='call_MRTmLfQ4i8lIEWVr0brw0pdI'
        ),
        ToolMessage(
            content='中方三艘油轮通过霍尔木兹海峡',
            name='get_news',
            id='8c566b8b-6c53-46b3-a468-b20070c347ae',
            tool_call_id='call_99pkvYtxXwpKLyPA7AlKp9SI'
        ),
        ToolMessage(
            content='邮件ID: sk2131421\n是空的',
            name='read_email_tool',
            id='4b24050e-c011-4e56-be41-08bdaa98e8e0',
            tool_call_id='call_dwtYMN8FNpx7T15H21CP8roh'
        ),
        ToolMessage(
            content='发送给 15641685664@qq.com 的邮件标题是: 哈哈哈, 内容: 你好啊',
            name='send_email_tool',
            id='2ef1a0f0-1491-424e-827e-4ecc757